# Online Retail · Segmentación RFM

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Nicolenki7/Online-Retail-Business-Analysis/blob/main/Online_Retail_Analysis.ipynb)

## La pregunta de negocio

Un minorista online tiene **4.338 clientes** y **8,89 millones** de facturación acumulada. Tratar a todos igual cuesta dinero: el cliente que compra una vez al año no merece el mismo esfuerzo que el que sostiene la caja. La pregunta es **quién sostiene la facturación, quién se está yendo y cuánto dinero hay en juego**.

Este cuaderno responde:

1. **Qué segmento** aporta la facturación, y si coincide con el más numeroso.
2. **Cuán concentrada** está la facturación: cuánto depende la empresa de unos pocos clientes.
3. **Cuánto dinero está en riesgo** de perderse con los clientes que se están enfriando.

**Datos.** Online Retail (UCI), 4.338 clientes con sus tres métricas RFM ya calculadas y su segmento asignado en `rfm_segmentation_final.csv`, versionado en el repo.


In [ ]:
import pathlib

import pandas as pd

try:
    from src.svg_chart import AZUL, ROJO, barras
except ImportError:  # en Colab el repo no está clonado
    import urllib.request
    url = ('https://raw.githubusercontent.com/Nicolenki7/'
           'Online-Retail-Business-Analysis/main/src/svg_chart.py')
    exec(urllib.request.urlopen(url).read().decode())

d = pd.read_csv('rfm_segmentation_final.csv')
total = d['Monetary'].sum()
print(f'{len(d):,} clientes · facturación {total:,.0f}')
print(d.head(3).to_string())


## 1. Qué significa cada letra

- **R (Recency):** días desde la última compra. Más bajo es mejor.
- **F (Frequency):** número de compras. Más alto es mejor.
- **M (Monetary):** gasto total. Más alto es mejor.

Cada métrica está puntuada en **cuartiles de 1 a 4**, así que las puntuaciones son **relativas a esta base de clientes**, no absolutas: un 4 en M significa estar en el cuarto que más gasta aquí, no gastar mucho en términos universales. Eso importa al leer el resto.


In [ ]:
g = d.groupby('Segment').agg(
    clientes=('CustomerID', 'size'),
    ingresos=('Monetary', 'sum'),
    recencia=('Recency', 'mean'),
    frecuencia=('Frequency', 'mean'),
    ticket=('Monetary', 'mean'))
g['pct_clientes'] = g.clientes / len(d) * 100
g['pct_ingresos'] = g.ingresos / total * 100
g = g.sort_values('pct_ingresos', ascending=False)
g.round(1)


## 2. El segmento que paga la factura

Los nombres completos no caben en el ancho de barra, así que los gráficos usan etiquetas cortas. El orden es el mismo en los dos gráficos, para que se puedan leer en paralelo: **Leales** = *Clientes Leales*, **En riesgo** = *Clientes en Riesgo*, **Potencial** = *Potencial Leal*, **No perder** = *No se pueden perder*.


In [ ]:
IMG = pathlib.Path('docs/img')
CORTO = {'Clientes Leales': 'Leales', 'Clientes en Riesgo': 'En riesgo',
         'Potencial Leal': 'Potencial', 'Otros': 'Otros',
         'No se pueden perder': 'No perder', 'Durmiendo': 'Durmiendo',
         'Nuevos Clientes': 'Nuevos'}

barras(
    [(CORTO.get(s, s), v, ROJO if v > 50 else AZUL)
     for s, v in g['pct_ingresos'].items()],
    'Un segmento sostiene la caja',
    IMG / 'rfm-ingresos.svg',
    subtitulo='% de la facturación total por segmento RFM',
    formato='{:.1f}', sufijo='%')

barras(
    [(CORTO.get(s, s), v, AZUL)
     for s, v in g['pct_clientes'].reindex(g.index).items()],
    'Y no es el segmento más numeroso',
    IMG / 'rfm-clientes.svg',
    subtitulo='% de los clientes por segmento (mismo orden que el gráfico anterior)',
    formato='{:.1f}', sufijo='%')


## 3. Cuánto depende la empresa de unos pocos

El segmento ya lo sugiere; el Pareto lo mide. Se ordenan los clientes de mayor a menor gasto y se calcula qué parte acumulada de la facturación aporta cada grupo.


In [ ]:
ordenado = d['Monetary'].sort_values(ascending=False).reset_index(drop=True)
acumulado = ordenado.cumsum() / total * 100

def cuota(pct):
    k = max(1, int(len(ordenado) * pct / 100))
    return acumulado.iloc[k - 1]

cortes = [1, 5, 10, 20, 50]
pareto = {f'top {c}%': cuota(c) for c in cortes}
print({k: round(v, 1) for k, v in pareto.items()})

barras(
    [(k, v, ROJO if c in (1, 10) else AZUL)
     for (k, v), c in zip(pareto.items(), cortes)],
    f"El 10 % de los clientes paga el {pareto['top 10%']:.0f} % de la factura",
    IMG / 'rfm-pareto.svg',
    subtitulo='% acumulado de la facturación que aporta cada grupo de clientes',
    formato='{:.1f}', sufijo='%')


In [ ]:
riesgo = d[d.Segment.isin(['Clientes en Riesgo', 'No se pueden perder'])]
print(f'En riesgo: {len(riesgo)} clientes ({len(riesgo)/len(d):.1%}) · '
      f'{riesgo.Monetary.sum():,.0f} ({riesgo.Monetary.sum()/total:.1%} de la facturación)')
print('recencia media:', round(riesgo.Recency.mean(), 1), 'vs resto:',
      round(d[~d.Segment.isin(['Clientes en Riesgo', 'No se pueden perder'])].Recency.mean(), 1))

pathlib.Path('outputs').mkdir(exist_ok=True)
g.round(2).to_csv('outputs/segment_summary.csv')


## Qué dicen los números

- **El 24,2 % de los clientes (1.050 Leales) aporta el 71,8 % de la facturación**: 6,38 de los 8,89 millones. El segmento **no** es el más numeroso — Durmiendo y Potencial tienen cada uno más gente (1.015 y 1.013) con el 2,9 % y el 7,1 % de los ingresos.
- **La concentración es extrema:** el 1 % de los clientes paga el **31,8 %** de la factura, y el 10 % paga el **61,4 %**. Perder a un puñado de clientes no es perder cuota: es perder un tercio del negocio.
- **En riesgo hay 606 clientes** (*En riesgo* + *No se pueden perder*): el 14,0 % de la base y **1,16 millones, el 13,1 % de la facturación**. Su recencia media es de 113,8 días frente a 89,1 del resto: llevan tiempo sin comprar y no son compradores de una sola vez.
- **El ticket medio separa los segmentos mejor que el conteo:** 6.077 por cliente Leal frente a 257 de quien duerme. No es lo mismo recuperar a un cliente de 6.000 que a uno de 250, aunque en un ranking por número de clientes aparezcan juntos.

### Limitaciones

- **Las puntuaciones RFM son relativas**: cuartiles sobre esta base, no umbrales absolutos. Añadir clientes nuevos cambia las puntuaciones de todos, así que dos ejecuciones separadas en el tiempo no son comparables sin recalcular.
- **`Monetary` es gasto acumulado sin fecha**: un cliente que gastó mucho hace dos años y otro que gasta igual este mes salen con la misma puntuación M. La R corrige parte, no todo.
- **No hay línea de tiempo ni producto**: el CSV no permite ver estacionalidad ni qué categorías empujan el gasto.
- **Los cortes de segmento son reglas de negocio**, no clústeres de un modelo: son explicables y accionables, pero también discutibles. Un k-means daría grupos distintos sobre los mismos datos.
- La facturación incluye devoluciones y cancelaciones del dataset original si el preprocesado no las eliminó: no se puede auditar desde este CSV.

### Siguiente paso

Pasar de segmento a valor esperado: cruzar la probabilidad de recompra de cada cliente con su ticket para decidir a quién merece la pena rescatar primero. Con el 13,1 % de la facturación en riesgo, el orden de la lista de llamadas vale más que el tamaño de la lista.
